In [1]:
print('되나? 좀 되라.. ')

되나? 좀 되라.. 


In [ ]:
## 이 노트북은 RAG를 테스트 용도입니다.
## 일반적으로 웹에서 사용하는 용도가 아닌, 프로그램을 만들어서 LLM모델을 사용하려고 한다
## LLM 모델은 : Gemini(google)-무료제공버전  참고:GPT(openai)-유료
## 컨셉 : 메뉴데이터를 벡터데이터베이스 구축 한 후 사용자 질의에 응답하는 챗봇

In [2]:
# llm 모델테스트, 
# 우리가 사용할 llm 모델은 구글 제미나이이니다.
# 그래서 구글ai studio에서 키를 발급 받아야 한다. 이 키를  api key
# 그리고 이 키를 로컬에서 관리합니다. 그래서 .env라는 폴더를 먼저 만들어요. 위치꼭 기억
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()  # .env 에 있는 구글 api 키를 자동으로 로딩

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
# response = llm.invoke('대한민국 소개해 줘')
# print(response.content)

# 모델이 정상 동작임을 확인했음..  RAG중 G부분에서 사용할 모델 테스트 완료 됨.

In [3]:
## RAG 처럼 테스해 볼께요
query = "자장면은 달고 짜면서 맛있어.   짬뽕은 얼큰하고 맛있어.. 이게 내가 주는 데이터야.. 이 데이터로 광고문구만들어줘"
response = llm.invoke(query)
print(response.content)

## llm이 만들어 준 응답으로 서비스를 제공한다..




[{'type': 'text', 'text': '제공해주신 데이터(**자장면: 달고 짜고 맛있다 / 짬뽕: 얼큰하고 맛있다**)를 바탕으로, 상황과 콘셉트에 맞게 활용하실 수 있는 광고 문구를 여러 가지 버전으로 작성해 드립니다.\n\n---\n\n### 1. 직관적이고 중독성 있는 짧은 카피 (배너, SNS용)\n> **"달고 짜서 미치고, 얼큰해서 지리는 맛!"**\n> 단짠의 정석 **자장면** vs 속 뚫리는 얼큰함 **짬뽕**\n> 고민할 시간에 둘 다 시키세요!\n\n### 2. 입맛을 자극하는 감성 카피 (메뉴판, 포스터용)\n> 입안 가득 감기는 **달콤·짭짤한 자장면의 유혹.**\n> 가슴속까지 풀리는 **얼큰하고 시원한 짬뽕의 매력.**\n> **어떤 걸 선택해도 최고의 맛입니다.**\n\n### 3. 유쾌하고 위트 있는 카피 (배달 앱 상단 문구용)\n> **단짠단짠 자장면**이냐, **얼큰칼칼 짬뽕**이냐!\n> 그것이 문제로다?\n> **아뇨, 둘 다 \'맛있다\'가 정답입니다.**\n\n### 4. 맛의 특징을 강조한 카피 (기본형)\n> **[달고 짠 미학, 자장면]**\n> 한 입 먹으면 멈출 수 없는 단짠의 환상 조합!\n>\n> **[얼큰한 국물, 짬뽕]**\n> 속이 뻥 뚫리는 얼큰함과 깊은 풍미!\n>\n> **오늘 당신의 완벽한 한 끼를 책임집니다.**\n\n---\n\n**💡 추천 활용 팁:**\n* **가장 반응이 좋은 키워드:** \'단짠\'(달고 짠), \'얼큰\'\n* **타겟:** 결정 장애가 있는 손님들에게 "둘 다 맛있으니 탕짜면/짬짜면을 먹거나 둘 다 시키라"는 유도를 하면 매출 상승에 도움이 됩니다.\n\n원하시는 특정 채널(인스타그램, 배달의민족, 현수막 등)이 있다면 그에 맞춰 문구를 더 수정해 드릴 수 있습니다!', 'extras': {'signature': 'Eq8aCqwaAWkUfRPJ62i0ShlkvvpZsRgQzBgu7vO0VOTCW77InouFVuVfSjJZubvQwXg1Bpmpt9Dv4OB

In [3]:
## 1. LLM에게 전달 할때 우리가게(음식가게)의 메뉴의 정보를 제공해야죠
## 그러기 위해서 .. 메뉴의 정보를 불러오고 > 벡터데이터로 구축해야 한다.
## 우리정보는 어디에 있나요?  menus_recommendation.json
import json
from langchain_core.documents import Document  
#벡터디비 만들때는 데이터(우리는 json)를 Document로 만들어야함
#Document 객체는 키의 이름이 page_content와  metadata 로 되어 있음 (단순 암기 영역)

#우리의 json 데이터 로드
with open("../storage/menus_recommendation.json", "r", encoding="utf-8") as f:
    data = json.load(f)
#print(data)
#실제로는 데이터가 몇 만건일 수 있다. 이래야지. 성능이 좋게 나온다.
#일부만 확인한다. 이유: 데이터가 정확하게 로딩되었는지 확인, 한글이 깨졌는지 확인 등
print(data[:1]) # 앞에서 부터 3개
print(len(data))

#데이터를 가져왔으니.. 벡터데이터베이스를 만들기 전에 Documents 문서 객체로 변환
# Document생성
docs=[] # document 문서를 저장할 배열
for row in data:
    #content변수는 문자열입니다. 이름:자장면 설명:자장면설명 이렇게 하나의 문자열로 만든다.
    content=f" 이름:{row['name']}  설명:{row['description']}"
    #print(content)
    docmake = Document(
        page_content=content.strip(),
        #  메타 데이터 추가 -------------------------------
        metadata={
                "id": int(row["id"]),
                "name": row["name"],
                "calories": int(row["calories"]),
                "recommendation": int(row["recommendation"]),
                "description": row["description"]
            }
    )
    docs.append(docmake)
print(docs[:2])  # 리스트에는 document 타입의 객체가 저장되어 있음

[{'id': 5, 'name': '마파두부', 'calories': 400, 'description': '두부와 다진 고기를 매콤한 소스로 볶은 중국 요리', 'recommendation': 92}]
97
[Document(metadata={'id': 5, 'name': '마파두부', 'calories': 400, 'recommendation': 92, 'description': '두부와 다진 고기를 매콤한 소스로 볶은 중국 요리'}, page_content='이름:마파두부  설명:두부와 다진 고기를 매콤한 소스로 볶은 중국 요리'), Document(metadata={'id': 6, 'name': '깐풍기', 'calories': 750, 'recommendation': 313, 'description': '튀긴 닭고기를 매콤달콤한 소스에 볶은 요리'}, page_content='이름:깐풍기  설명:튀긴 닭고기를 매콤달콤한 소스에 볶은 요리')]


In [4]:
# 위에서 생성한  documnet 가 저장된  docs리스트로 벡터 데이터 베이스를 생성해야 한다.
# 이때 docs문장에 있는 한글을 숫자화 해야 합니다.
# 가장 좋은 성능을 보이는 트랜스포머로 만든 모델을 사용할 것입니다.
# 한글을 벡터화 시키는 것을 임베딩모델이라고 하고,  우리가 사용할 임베딩 모델은
# 트랜스포머 방식으로 개발된 모델입니다.
from langchain_huggingface import HuggingFaceEmbeddings

#jhgan/ko-sroberta-multitask 이 임베딩 모델을 다운로드 받는 과정..
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},  # GPU 있으면 "cuda"
    encode_kwargs={"normalize_embeddings": True} # 벡터 비교에서 중요한 건 “방향”임
)

d:\dev\2026\menus_project\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1411.37it/s]


In [5]:
## 벡터데이터베이스를 구축한다
## rag 에서 사용하는 벡터데이터베이스는 정형데이터베이스랑 다르다.
## 종류는  pincorn  faiss   chroma 디비등이 있다. 우리는 chroma db를 사용하겠다.
## 그래서 디비가 저장된 디렉토리를 생성함 .. chroma_db

## 벡터베이터베이스 생성
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(
    documents=docs,    # 벡터 데이터 베이스를 구축할 정보
    embedding=embeddings,  # docs의 문장을 해석하여 데이터베이스를 구축할 임베딩 객체
    persist_directory="../chroma_db/menus_db_meta"  # 데이터베이스 위치
)

In [18]:
## 벡터데이터베이스 구축된 정보 확인
print("저장된 문서 개수 :", vectorstore._collection.count())

저장된 문서 개수 : 97


In [6]:
## 벡터데이터베이스를 한 번 구축하면 다음에는 구축안해도 되고 읽어 오면 됩니다.
## 이미 구축된 데이터베이스에서 데이터를 읽오 오는 법
def loadVectorStore():
    return Chroma(
        persist_directory="../chroma_db/menus_db_meta",
        embedding_function=embeddings  # 주의할 점.. 벡터디비구축시 embedding 모델과
                                    # 읽어 올 때의 모델은 동일해야 합니다. 
    )

In [8]:
vectorstore

In [7]:
## 지금까지의 작업으로 선정한 데이터(json)으로 벡터데이터베이스를 구축함.
## 사용자 질의로 벡터데이터베이스와 유사도가 높은 문서를 검색해보자

#사용자질문
query = "매콤한 음식 추천"
results = vectorstore.similarity_search(query, k=4)

for doc in results:
    print(doc.page_content)


이름:양념치킨  설명:튀긴 닭고기에 매콤달콤한 양념을 입힌 음식
이름:닭갈비  설명:닭고기와 채소를 매콤한 양념에 볶은 음식
이름:제육볶음  설명:돼지고기를 매콤한 양념에 볶은 음식
이름:떡볶이  설명:떡을 매콤달콤한 고추장 양념에 끓인 음식


In [21]:
# [핵심 1] 사용자 질문을 분석하여 검색어와 메타데이터 필터 조건을 동적으로 추출
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
def analyze_query(user_query):
    analysis_prompt = ChatPromptTemplate.from_template("""
    사용자의 질문을 분석하여 아래 JSON 형식으로만 결과를 출력해줘. (다른 설명은 절대 붙이지 말 것)
    - query: 벡터 검색을 위한 핵심 키워드 또는 메뉴 이름 (예: "매운 요리", "마라탕", "김치찌개")
    - max_calories: 질문에 언급된 최대 칼로리 조건 (숫자만 추출, 없으면 null)
    - min_recommendation: 질문에 언급된 최소 추천수 조건 (숫자만 추출, 없으면 null)

    [사용자 질문]: {query}
    
    [출력 형식 (JSON)]:
    {{"query": "추출된검색어", "max_calories": null, "min_recommendation": null}}
    """)
    
    chain = analysis_prompt | llm | StrOutputParser()
    result_str = chain.invoke({"query": user_query})
    
    try:
        print(result_str)
        # LLM 출력이 마크다운 코드 블록으로 감싸져 있을 경우 처리
        #cleaned = result_str.replace("```json", "").replace("```", "").strip()
        return json.loads(result_str)
    except Exception as e:
        # 파싱 실패 시 원본 질문을 검색어로 사용
        return {"query": user_query, "max_calories": None, "min_recommendation": None}

#analyze_query("매운 면 요리 추천")
userquery = "매운 면 요리인네 칼로리는 최대 1000이야";
analyzed = analyze_query(userquery)
search_q = analyzed['query']
max_cal = analyzed.get("max_calories")
min_rec = analyzed.get("min_recommendation")
print(search_q, max_cal, min_rec)

{"query": "매운 면 요리", "max_calories": 1000, "min_recommendation": null}
매운 면 요리 1000 None


In [ ]:
# [핵심 2] 추출된 조건(max_calories, min_recommendation)을 Chroma 필터에 동적 적용
# $lte (Less Than or Equal): 이하 
# $lt (Less Than): 미만 
# $gte (Greater Than or Equal): 이상
# $gt (Greater Than): 초과 

def retrieval_docs(search_query, max_calories=None, min_recommendation=None, k=5):
    vectorstore = loadVectorStore()
    
    filter_conditions = {}
    if max_calories is not None:
        filter_conditions["calories"] = {"$lte": int(max_calories)}
    if min_recommendation is not None:
        filter_conditions["recommendation"] = {"$gte": int(min_recommendation)}
        
    search_filter = filter_conditions if filter_conditions else None
    
    print(f"📌 [검색 적용] 쿼리: '{search_query}' | 메타데이터 필터: {search_filter}")
    
    results = vectorstore.similarity_search(
        search_query, 
        k=k, 
        filter=search_filter
    )
    return results

userquery = "매운 면 요리인네 칼로리는 최대 1000이야";
analyzed = analyze_query(userquery)
search_q = analyzed.get("query")
max_cal = analyzed.get("max_calories")
min_rec = analyzed.get("min_recommendation")
retrieval_docs(search_q, max_cal, min_rec)

📌 [검색 적용] 쿼리: '매운 면 요리' | 메타데이터 필터: {'calories': {'$lte': 1000}}


[Document(id='19c1cef8-30a5-4891-be51-7d56b43b1e99', metadata={'description': '쫄깃한 면을 매콤한 양념에 비빈 음식', 'recommendation': 226, 'name': '쫄면', 'calories': 500, 'id': 36}, page_content='이름:쫄면  설명:쫄깃한 면을 매콤한 양념에 비빈 음식'),
 Document(id='b7867333-0340-47ff-9aa3-1ddb8262ea5f', metadata={'recommendation': 245, 'calories': 450, 'id': 31, 'description': '소면을 매콤새콤한 양념에 비빈 음식', 'name': '비빔국수'}, page_content='이름:비빔국수  설명:소면을 매콤새콤한 양념에 비빈 음식'),
 Document(id='11eafda9-22d6-4eba-8377-3fa5e67cae15', metadata={'id': 34, 'recommendation': 463, 'description': '차가운 육수나 양념에 면을 넣은 음식', 'name': '냉면', 'calories': 450}, page_content='이름:냉면  설명:차가운 육수나 양념에 면을 넣은 음식'),
 Document(id='9ddbb31d-dbfa-4f15-ab83-792874c2c8bb', metadata={'id': 55, 'calories': 650, 'description': '넓은 파스타 면과 소스를 층층이 쌓아 구운 요리', 'recommendation': 166, 'name': '라자냐'}, page_content='이름:라자냐  설명:넓은 파스타 면과 소스를 층층이 쌓아 구운 요리'),
 Document(id='0c274713-7f24-4d89-b647-4372a517eb18', metadata={'recommendation': 287, 'id': 29, 'description': '라면과 떡을 매콤한 양

In [ ]:
## 이 셀의 목적은 유사도 검색의 정확도를 분석해서 품질을 높이기 위한 테스트
## 그래서 스코어를 확인하기 위해서 similarity_search_with_score메서드를 사용함
## similarity_search메서드로 검색했는데
## similarity_search_with_score메서드로 검색해 보겠다.. (유사도까지 리턴)

def search_menus(quere, k=3):
    results = vectorstore.similarity_search_with_score(query, k=k)
    output=[]
    for doc, score in results:
        output.append({
            "내용":doc.page_content,
            "유사도":score
        })
    return output

##  테스트
query =" 만두를 바삭하게 구운 음식 소개해 줘"
results = search_menus(query,4)
for r in results:
    print('유사도', r['유사도'])
    print(r['내용'])
    print('---------------------')


# 0.0 완전히동일,  0.2 매우 유사    1.0 보통    5.0 전혀 안 비슷

유사도 0.33699971437454224
이름:군만두 
       설명:속을 채운 만두를 바삭하게 구운 음식
---------------------
유사도 0.6472766995429993
이름:만둣국 
       설명:만두를 따뜻한 국물에 넣어 끓인 음식
---------------------
유사도 0.7153041362762451
이름:치킨 
       설명:닭고기를 바삭하게 튀기거나 구워 만든 음식
---------------------
유사도 0.7393913865089417
이름:도넛 
       설명:밀가루 반죽을 튀겨 만든 달콤한 빵
---------------------


In [ ]:
## 위의 코드는 데이터베이스에서 가져온 값입니다.
## 포인트는 유사도 검색을 해서 가져왔다는 것..  코사인유사도.. 벡터테이터베이스 구축이 되어야 함.
## 벡터데이터베이스로 구축하려면 한글을 쪼개서 분석한다. 이 모델이 임베딩모델..(트랜스포머 기반으로 개발된)
## 개발자는 모델을 개발하겠느냐.. 이미 개발된 모델로 제품을 만드는가?  노선을 정하고 공부..

## 컨셉  데이터베이스는 회사의 자료입니다..  이것을 벡터디비로 구축해성
## if문등으로 동일한 문장이 아닌 .. 유사한 문장을 찾아 내는 방식..
## 검색하는 입장에서는 매우 편하겠다..

## 이제는 이 문장을 그대로 사용자에게 보여 줄 것인가? 아니면 llm 모델을 사용해서
## 회사가 원하는 컨셉으로 문장을 다시 만들 것인가? 이 판단이죠
## 컨셉은 유사도 검색한 문장과 사용자 질문으로 llm 에게 문장을 만들어 달라고 할 예정

In [ ]:
## RAG에서 A 단계
def format_docs_with_metadata(docs):
    if not docs:
        return "문서가 없다. 검색된 문서가 없다"

    formatted_list= []

    for doc in docs:
        meta =doc.metadata
        info = f"""
        [메뉴정보]
        - 칼로리:{meta['calories']}
        - 이름:{meta['name']}
        - 추천수:{meta['recommendation']}
        [문서내용]
        {doc.page_content}
        """
        formatted_list.append(info)

    return "\n\n".join(formatted_list)   #join함수는 리스트에 있는 모든 값을 하나의 문자열로 합쳐라..


#테스트
userquery = "매운 면 요리인데 칼로리가 최대 1000인 음식"
results_ana = analyze_query(userquery)
query = results_ana['query']
max_cal =results_ana['max_calories']
min_rec = results_ana['min_recommendation']

#R
rdocs = retrieval_docs(query, max_cal, min_rec)   # 노트북에서는 마지막 코드를 출력해서 보여줘요
#A
format_docs_with_metadata(rdocs)

In [23]:
def rag_process(user_query):
    # 1. 질문 분석 단계 (질문에서 의도, 칼로리 조건, 검색어 분리)
    analyzed = analyze_query(user_query)
    search_q = analyzed.get("query")
    max_cal = analyzed.get("max_calories")
    min_rec = analyzed.get("min_recommendation")
    
    # 2. 동적 필터와 검색어를 반영해 문서 검색 수행
    docs = retrieval_docs(search_q, max_calories=max_cal, min_recommendation=min_rec)
    context = format_docs_with_metadata(docs)

    prompt = ChatPromptTemplate.from_template("""
    너는 음식 추천 전문가야. 아래의 [참고 정보 (메타데이터 기반)]를 바탕으로 사용자 질문에 알맞은 메뉴를 1. 2. 형식으로 대답해 줘.
    
    [사용자 질문] 
    {query}
    
    [참고 정보 (메타데이터 기반)]
    {context} 
    
    [규칙]
    - 제공한 참고 정보의 메타데이터(칼로리, 추천수 등)를 답변에 적극적으로 반영할 것
    - 음식이름을 먼저 이야기할 것
    - 2줄 정도 요약해 줄 것
    - 친절하게 구매하거나 선택할 수 있도록 이야기해 줄 것             
    """)

    chain = prompt | llm | StrOutputParser()
    result = chain.invoke({'query': user_query, 'context': context})
    return result

userquery = "매운 면 요리인네 칼로리는 최대 1000이야";
rag_process(userquery)

{"query": "매운 면 요리", "max_calories": 1000, "min_recommendation": null}
📌 [검색 적용] 쿼리: '매운 면 요리' | 메타데이터 필터: {'calories': {'$lte': 1000}}


"안녕하세요! 요청하신 **'1000 kcal 이하의 매운 면 요리'** 조건에 딱 맞는 맛있는 메뉴들을 추천해 드릴게요.\n\n1. **쫄면**\n쫄깃한 면발을 매콤한 양념에 비벼 먹는 요리로, 500 kcal라는 부담 없는 칼로리로 즐기실 수 있어요.\n226회의 많은 추천을 받은 검증된 별미이니, 씹는 맛과 매콤함이 당기실 때 꼭 선택해 보세요!\n\n2. **비빔국수**\n소면에 매콤새콤한 양념이 어우러져 가볍고 입맛을 돋우기에 최고인 요리입니다.\n칼로리가 450 kcal로 매우 낮고 245회의 추천을 받은 인기 메뉴이니, 부담 없이 깔끔하게 즐겨보시길 추천해 드려요.\n\n3. **라볶이**\n라면과 떡을 매콤한 양념에 볶아내어 중독성 있는 매운맛을 자랑하는 든든한 요리입니다.\n600 kcal로 1000 kcal 기준을 넉넉하게 충족하며, 287회나 추천된 만큼 만족스러운 한 끼 식사로 후회 없는 선택이 되실 거예요."